# 00_setup_config.ipynb - Setup & Konfigurasi LURGIP MVP

Notebook ini untuk:
- Verifikasi koneksi ke 3 port database
- Test import semua modul
- Validasi struktur folder

**ATURAN:** Notebook ini idempotent - bisa dijalankan ulang dari awal.

In [ ]:
# ── IMPORT LIBRARIES ─────────────────────────────────────────────────────
import sys
from pathlib import Path

# Tambahkan src ke path
sys.path.insert(0, str(Path.cwd() / "src"))

print("✅ Path setup complete")

In [ ]:
# ── TEST IMPORT SEMUA MODUL ─────────────────────────────────────────────
try:
    from config import DB_CONFIGS, PORT_NAMESPACE, DEPO_TO_PORT
    from config import GHOST_DAYS, CHURN_THRESHOLD, VISIT_DURATION_MAX
    from config import DATA_DIR, REF_DIR, OUT_DIR
    print("✅ config.py imported successfully")
except Exception as e:
    print(f"❌ Error importing config: {e}")

try:
    from db import get_connection, query_to_df, union_ports
    from db import add_port_namespace, normalize_all_datetimes
    print("✅ db.py imported successfully")
except Exception as e:
    print(f"❌ Error importing db: {e}")

try:
    from geo_validator import parse_coords, flag_ghost_coords, clean_coords
    print("✅ geo_validator.py imported successfully")
except Exception as e:
    print(f"❌ Error importing geo_validator: {e}")

try:
    from analysis import detect_ghost_outlets, calc_route_compliance
    from analysis import visit_duration_summary, sales_performance
    from analysis import calc_prospect_potential
    print("✅ analysis.py imported successfully")
except Exception as e:
    print(f"❌ Error importing analysis: {e}")

In [ ]:
# ── VERIFIKASI STRUKTUR FOLDER ──────────────────────────────────────────
print("=== STRUKTUR FOLDER ===")
for folder in [DATA_DIR, REF_DIR, OUT_DIR]:
    status = "✅ exists" if folder.exists() else "❌ missing"
    print(f"{folder}: {status}")

# Cek isi folder reference
print("\n=== FOLDER REFERENCE ===")
if REF_DIR.exists():
    files = list(REF_DIR.iterdir())
    if files:
        for f in files:
            print(f"  📄 {f.name}")
    else:
        print("  ⚠️  Folder kosong - silakan upload:")
        print("     - MASTER OUTLET AQUA.xlsx")
        print("     - RUTE ALL.xlsx")
        print("     - depo_coords.json (opsional)")

In [ ]:
# ── TEST KONEKSI DATABASE ───────────────────────────────────────────────
import pymysql

print("=== TEST KONEKSI DATABASE ===")

for port_key, config in DB_CONFIGS.items():
    try:
        conn = get_connection(port_key)
        with conn.cursor() as cur:
            cur.execute("SELECT DATABASE() AS db_name")
            result = cur.fetchone()
            print(f"✅ Port {config['port']}: Connected to '{result['db_name']}'")
        conn.close()
    except Exception as e:
        print(f"❌ Port {config['port']}: Connection failed - {e}")

In [ ]:
# ── VALIDASI SCHEMA DATABASE ────────────────────────────────────────────
# Cek apakah tabel-tabel penting ada
REQUIRED_TABLES = [
    "sfa_doccallitem",
    "sfa_gpstracking",
    "dms_sm_saledtl",
    "dms_sm_addressinfo",
    "mst_customer",
    "mst_employee"
]

print("=== VALIDASI TABEL DATABASE (Port 3306) ===")

conn = get_connection("port_3306")
with conn.cursor() as cur:
    cur.execute("SHOW TABLES")
    tables = [row[f"Tables_in_{DB_CONFIGS['port_3306']['database']}"] 
              for row in cur.fetchall()]

for table in REQUIRED_TABLES:
    status = "✅" if table in tables else "❌"
    print(f"{status} {table}")

conn.close()

In [ ]:
# ── CEK KOLOM PENTING (Rule #2, #3, #4) ─────────────────────────────────
print("=== VALIDASI NAMA KOLOM KRITIS ===")

conn = get_connection("port_3306")

# Cek sfa_doccallitem - harus ada szLangitude (TYPO!)
with conn.cursor() as cur:
    cur.execute("SHOW COLUMNS FROM sfa_doccallitem")
    cols = [row["Field"] for row in cur.fetchall()]
    
print("\n📋 sfa_doccallitem:")
print(f"  {'✅' if 'szLangitude' in cols else '❌'} szLangitude (TYPO - jangan dibetulkan!)")
print(f"  {'✅' if 'decDuration' in cols else '❌'} decDuration (dalam DETIK)")
print(f"  {'✅' if 'dtDocCall' in cols else '❌'} dtDocCall")

# Cek sfa_gpstracking - harus ada szLangitude (TYPO!)
with conn.cursor() as cur:
    cur.execute("SHOW COLUMNS FROM sfa_gpstracking")
    cols = [row["Field"] for row in cur.fetchall()]

print("\n📋 sfa_gpstracking:")
print(f"  {'✅' if 'szLangitude' in cols else '❌'} szLangitude (TYPO)")
print(f"  {'✅' if 'dtTimestamp' in cols else '❌'} dtTimestamp")

# Cek dms_sm_addressinfo - harus ada szLatitude (tanpa typo)
with conn.cursor() as cur:
    cur.execute("SHOW COLUMNS FROM dms_sm_addressinfo")
    cols = [row["Field"] for row in cur.fetchall()]

print("\n📋 dms_sm_addressinfo:")
print(f"  {'✅' if 'szLatitude' in cols else '❌'} szLatitude (tanpa typo)")
print(f"  {'✅' if 'szLongitude' in cols else '❌'} szLongitude")

conn.close()

In [ ]:
# ── RINGKASAN KONFIGURASI ───────────────────────────────────────────────
print("\n" + "="*60)
print("✅ SETUP CONFIG SELESAI")
print("="*60)
print(f"\n📊 Database: {DB_CONFIGS['port_3306']['host']}:{DB_CONFIGS['port_3306']['port']}")
print(f"🗂️  Namespace: {PORT_NAMESPACE}")
print(f"👻 Ghost threshold: {GHOST_DAYS} hari")
print(f"⚠️  Churn threshold: {CHURN_THRESHOLD*100:.0f}%")
print(f"⏱️  Max visit duration: {VISIT_DURATION_MAX} detik ({VISIT_DURATION_MAX/60:.1f} menit)")
print("\n🚀 Lanjut ke notebook 01_extract_data.ipynb")